# 02 – Convolutional Neural Networks (CNN)

CNNs are the backbone of computer vision tasks.

Topics covered:
1. Convolution, Pooling, and Padding intuition
2. Building a CNN with Keras
3. Data Augmentation
4. Transfer Learning with a pretrained model (MobileNetV2)
5. Grad-CAM visualisation

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow:', tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

## 1. CIFAR-10 Dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# Normalise
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0
y_train = y_train.ravel()
y_test  = y_test.ravel()

print('Train:', X_train.shape, '  Test:', X_test.shape)

# Visualise some samples
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[i])
    ax.set_title(class_names[y_train[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 Samples'); plt.tight_layout(); plt.show()

## 2. Custom CNN

In [ ]:
def build_cnn():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.4),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn = build_cnn()
cnn.summary()

## 3. Data Augmentation

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1)
], name='augmentation')

# Visualise augmented images
sample_img = X_train[:1]
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for ax in axes:
    aug = augment(sample_img, training=True)[0]
    ax.imshow(aug); ax.axis('off')
plt.suptitle('Augmented Images'); plt.tight_layout(); plt.show()

In [ ]:
# Train with augmentation using tf.data
BATCH = 128
AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(10000)
    .batch(BATCH)
    .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

history = cnn.fit(
    train_ds, validation_data=val_ds,
    epochs=50,
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=1
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ['loss', 'accuracy']):
    ax.plot(history.history[m],         label='Train')
    ax.plot(history.history[f'val_{m}'], label='Val')
    ax.set_title(m.capitalize()); ax.legend()
plt.tight_layout(); plt.show()

## 4. Transfer Learning – MobileNetV2

In [ ]:
# Resize CIFAR-10 to 96x96 for MobileNetV2 (needs >= 32x32 but 96 is cleaner)
def preprocess_tl(x, y):
    x = tf.image.resize(x, (96, 96))
    x = keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
    return x, y

train_tl = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(10000).batch(BATCH)
    .map(preprocess_tl, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
)
val_tl = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH).map(preprocess_tl, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
)

base_model = keras.applications.MobileNetV2(input_shape=(96, 96, 3),
                                             include_top=False, weights='imagenet')
base_model.trainable = False   # freeze backbone

inputs  = keras.Input(shape=(96, 96, 3))
x_tl    = base_model(inputs, training=False)
x_tl    = layers.GlobalAveragePooling2D()(x_tl)
x_tl    = layers.Dropout(0.2)(x_tl)
outputs = layers.Dense(10, activation='softmax')(x_tl)
tl_model = keras.Model(inputs, outputs)

tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])

hist_tl = tl_model.fit(train_tl, validation_data=val_tl,
                       epochs=10, verbose=1)

print('\nTransfer Learning Test Accuracy:', tl_model.evaluate(val_tl, verbose=0)[1]:.3f)

## 5. Key Takeaways

| Concept | Notes |
|---------|-------|
| Conv2D | Learns spatial patterns with local filters |
| MaxPooling | Reduces spatial size; adds translation invariance |
| BatchNorm | Stabilises activations; allows higher learning rates |
| Data Augmentation | Artificially increases effective dataset size |
| Transfer Learning | Reuse pretrained features; only fine-tune top layers |

**Next:** `03_Ensemble_Methods.ipynb`